# E-Commerce Personalization Platform - Data Pipeline Notebook

## 1. Data Fetching and Initial Inspection
This section retrieves the UCI Online Retail dataset (ID 352) using the ucimlrepo library, loads the full original DataFrame (including InvoiceNo for RFM), and performs initial checks on structure, content, and quality. This helps identify issues like missing values (~25% in CustomerID) and negatives in Quantity/UnitPrice before cleaning

In [ ]:
from ucimlrepo import fetch_ucirepo

# Fetch dataset
online_retail = fetch_ucirepo(id=352)

# Use the original DataFrame to include all columns (InvoiceNo, StockCode, etc.)
df = online_retail.data.original.copy()

# Verify data and perform initial inspection
print("Columns in df:", df.columns.tolist())  # Confirm all 8 columns, including InvoiceNo
print(f'Shape: {df.shape}\n')  # Expected: (541909, 8)
print(f"First 5 Rows:\n{df.head()}\n")  # Verify loading with all columns
print(f"Data Info:\n")
df.info()  # Show dtypes and null counts
print(f"Summary Statistics:\n{df.describe()}")  # Basic stats for numeric columns
print(f"Missing Values:\n{df.isnull().sum()}")  # Check for nulls (expect ~25% in CustomerID)
print(f"\nDate Range (raw): {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")  # Before datetime conversion

# Metadata and variable information for reference
print("\nMetadata:")
print(online_retail.metadata)
print("\nVariable Information:")
print(online_retail.variables)

## 2. Data Visualization
Generate initial plots to explore distributions and patterns in the raw dataset (before cleaning) using Seaborn and Matplotlib. This includes histograms/boxplots for Quantity/UnitPrice (to detect outliers/negatives), top countries/products, and a correlation heatmap; helping guide cleaning decisions.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Quantity distribution (histogram)
plt.figure(figsize=(10, 6))
sns.histplot(df['Quantity'], bins=50)
plt.title('Distribution of Quantity (Raw Data)')
plt.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
plt.xlabel('Quantity')
plt.ylabel('Count')
plt.show()

# UnitPrice distribution (histogram)
plt.figure(figsize=(10, 6))
sns.histplot(df['UnitPrice'], bins=50)
plt.title('Distribution of UnitPrice (Raw Data)')
plt.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
plt.xlabel('UnitPrice')
plt.ylabel('Count')
plt.show()

# Boxplot for Quantity (outlier check)
plt.figure(figsize=(10, 6))
sns.boxplot(x=df['Quantity'])
plt.xscale('log')
plt.title('Box Plot of Quantity (Raw Data)')
plt.show()

# Boxplot for UnitPrice (outlier check)
plt.figure(figsize=(10, 6))
sns.boxplot(x=df['UnitPrice'])
plt.xscale('log')
plt.title('Box Plot of UnitPrice (Raw Data)')
plt.show()

# Top 10 countries by transaction count
plt.figure(figsize=(10, 6))
sns.countplot(y='Country', data=df, order=df['Country'].value_counts().index[:10])
plt.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
plt.title('Top 10 Countries by Transaction Count (Raw Data)')
plt.show()

# Additional: Top 10 products by frequency
print("Top 10 Products by Frequency (Raw Data):")
print(df['Description'].value_counts().head(10))

# Additional: Outlier min/max checks (printed for reference)
print(f"\nQuantity Outliers (min/max): {df['Quantity'].min()} / {df['Quantity'].max()}")
print(f"UnitPrice Outliers (min/max): {df['UnitPrice'].min()} / {df['UnitPrice'].max()}")

# Temporary TotalSpend for correlation heatmap (will recalculate after cleaning)
df_temp = df.copy()
df_temp['TotalSpend'] = df_temp['Quantity'] * df_temp['UnitPrice']
plt.figure(figsize=(10, 6))
sns.heatmap(df_temp[['Quantity', 'UnitPrice', 'TotalSpend']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap (Raw Data)')
plt.show()

## 3. Data Cleaning and Preprocessing
Perform data cleaning as per the project plan: remove cancellations (Quantity <= 0), drop rows with missing CustomerID (~25%), filter negative UnitPrice, convert InvoiceDate to datetime, and calculate TotalSpend. Verify changes with stats and save as 'data/cleaned_online_retail.csv' for further analysis.

In [ ]:
import pandas as pd

# Work with a copy to avoid warnings
df_clean = df.copy()

# Cleaning steps
df_clean = df_clean[df_clean['Quantity'] > 0] # Remove cancellations
df_clean = df_clean.dropna(subset=['CustomerID']) # Drop rows with null customerID (~25%)
df_clean = df_clean[df_clean['UnitPrice'] >= 0] # Remove negative UnitPrices (if any)

# Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], format='%m/%d/%Y %H:%M', errors='coerce', dayfirst=False)

# Verify conversion
print(f"Date Range after cleaning: {df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}")

# Calculate TotalSpend after cleaning to exclude negatives
df_clean['TotalSpend'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Verify cleaning
print(f"Shape after cleaning: {df_clean.shape}")
print(f"Missing Values after cleaning:\n{df_clean.isnull().sum()}")
print(f"Quantity min/max after cleaning: {df_clean['Quantity'].min()} / {df_clean['Quantity'].max()}")
print(f"UnitPrice min/max after cleaning: {df_clean['UnitPrice'].min()} / {df_clean['UnitPrice'].max()}")
print(f"TotalSpend min/max after cleaning: {df_clean['TotalSpend'].min()} / {df_clean['TotalSpend'].max()}")

# Save the cleaned DataFrame to a local CSV file
df_clean.to_csv('data/clean_online_retail.csv', index=False)
print("Data saved locally as 'data/cleaned_online_retail.csv'")

# Load and check the saved file
df_check = pd.read_csv('data/clean_online_retail.csv')
print("First 5 rows of saved cleaned data:")
print(df_check.head())

## 4. RFM Features Calculation and Post-Cleaning Verification
Calculate RFM (Recency, Frequency, Monetary) features using the cleaned DataFrame. This aggregates customer data for segmentation. Also, perform post-cleaning EDA checks and visualizations to verify the cleaning (e.g., re-plot histograms and boxplots, check distributions).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime

# Load the cleaned data if not already in memory (from previous section)
# df_clean = pd.read_csv('data/cleaned_online_retail.csv')  # Uncomment if restarting notebook

# Post-cleaning verification: Re-plot histograms and boxplots
for col in ['Quantity', 'UnitPrice']:
    plt.figure(figsize=(10, 6))
    sns.histplot(df_clean[col], bins=50, log_scale=True)
    plt.title(f'Distribution of {col} (After Cleaning)')
    plt.xlabel(f'log({col})')
    plt.ylabel('Count')
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    plt.show()

for col in ['Quantity', 'UnitPrice']:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df_clean[col])
    plt.xscale('log')
    plt.title(f'Box Plot of {col} (After Cleaning)')
    plt.show()

# Correlation heatmap after cleaning (with TotalSpend)
plt.figure(figsize=(10, 6))
sns.heatmap(df_clean[['Quantity', 'UnitPrice', 'TotalSpend']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap (After Cleaning)')
plt.show()

# Transactions over time (after datetime conversion)
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')
plt.figure(figsize=(12, 6))
sns.countplot(x='YearMonth', data=df_clean)
plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
plt.title('Transactions Over Time (After Cleaning)')
plt.xticks(rotation=45)
plt.show()

# Total Spend by Top 10 Countries (after cleaning)
country_spend = df_clean.groupby('Country')['TotalSpend'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
country_spend.plot(kind='barh')
plt.ticklabel_format(style='sci', axis='x', scilimits=(0,0))
plt.title('Total Spend by Top 10 Countries (After Cleaning)')
plt.xlabel('Total Spend')
plt.ylabel('Country')
plt.show()

# Calculate RFM features
today = df_clean['InvoiceDate'].max() + datetime.timedelta(days=1)
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (today - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',  # Frequency (unique invoices)
    'TotalSpend': 'sum'  # Monetary
}).reset_index() # this keeps CustomerID as column

# Rename columns
rfm = rfm.rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalSpend': 'Monetary'})

# Verify RFM
print("RFM DataFrame:")
display(rfm.head())
print("\nRFM Summary Statistics:")
display(rfm.describe())

# Plot RFM distributions
for col in ['Recency', 'Frequency', 'Monetary']:
    plt.figure(figsize=(10, 6))
    sns.histplot(rfm[col], bins=50, log_scale=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(f'log({col})')
    plt.ylabel('Count')
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    plt.show()

# Save RFM features
rfm.to_csv('data/rfm_features.csv', index=False)
print("RFM features saved as 'data/rfm_features.csv'")

## 5. Data Splitting
Split the cleaned dataset and RFM into train (70%), validation (15%), and test (15%) sets. This prepares for modeling (e.g., val for hyperparameter tuning, test for unbiased evaluation). Saves as CSVs in 'data/' for later days.

In [ ]:
from sklearn.model_selection import train_test_split

# Train/Val/Test split (70,15,15) for the cleaned data and RFM, preparing for future modeling
train_val, test = train_test_split(df_clean, test_size=0.15, random_state=42)
train, val = train_test_split(train_val, test_size=0.1765, random_state=42) # Approx 15% of original for val

print(f"Train shape: {train.shape}")
print(f"Validation shape: {val.shape}")
print(f"Test shape: {test.shape}")

# Optionally split RFM
rfm_train_val, rfm_test = train_test_split(rfm, test_size=0.15, random_state=42)
rfm_train, rfm_val = train_test_split(rfm_train_val, test_size=0.1765, random_state=42)

print(f"RFM Train shape: {rfm_train.shape}")
print(f"RFM Validation shape: {rfm_val.shape}")
print(f"RFM Test shape: {rfm_test.shape}")

# Save splits
train.to_csv('data/train.csv', index=False)
val.to_csv('data/val.csv', index=False)
test.to_csv('data/test.csv', index=False)
rfm_train.to_csv('data/rfm_train.csv', index=False)
rfm_val.to_csv('data/rfm_val.csv', index=False)
rfm_test.to_csv('data/rfm_test.csv', index=False)
print("Train, val, test, and RFM splits saved to 'data/' folder")

## 6. Data-Pipeline Summary
Completed Day 2 pipeline: Dataset fetched (541k rows), EDA revealed skew/outliers/negatives, cleaning reduced to ~398k rows, RFM aggregated for 4,339 customers (skewed Monetary/Frequency), and splits prepared. Insights: UK dominates (~90% transactions); prepare for bias in clustering. Ready for Day 3.

In [ ]:
# In a cell
from my_library.pipeline import fetch_and_inspect_data
df = fetch_and_inspect_data()  # Should run without error